# Calculating the rate of kinds of conversations in the massive english dataset.

In [1]:
import os
import numpy as np
import pandas as pd

from scipy.stats import skewtest, chisquare, t

# stats in R
import rpy2.robjects as ro
from rpy2.robjects import Formula
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri, default_converter, conversion

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../english/4-POWER-LAW-SCALING/reports'
candor_meta_data_path = '../english/data/all-candor-meta_data.csv'


parameter_values = os.path.join(DATA_PATH,'study-level-statistics.csv')
alpha_values = os.path.join(DATA_PATH, 'all.csv')

In [3]:
df = pd.read_csv(alpha_values)
print(df.shape)
df.head()

(23894, 7)


,corpus,cid,speaker,self,a,b,k
0,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,False,2.240313e-04,-0.727759,1066
1,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,True,1.439414e-08,0.363082,136
2,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,False,1.320233e-03,-0.615558,3107
3,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,True,1.493614e-03,-0.797152,2556
4,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS006,False,5.985318e-04,-0.503047,2966


In [4]:
dfp = pd.read_csv(parameter_values)
dfp = dfp.rename(columns={'b': 'b_'})
dfp.head()

,self,b_,std,k,se,tstat,p
0,False,-0.387600,0.943652,12019,0.008608,-45.030446,0.0
1,True,-0.486513,1.104611,10719,0.010669,-45.599785,0.0


In [5]:
df = pd.merge(
    left=df,left_on=['self'],
    right=dfp[['self', 'b_']], right_on=['self'],
    how='left'
)

In [6]:
df['faster'] = df['b'] < df['b_']

## Hypothesis test: Conversation Types are distributed equally.

We wanted to test the assumption that all possible combinations of information scaling factors are equal. We thus tested

$(\alpha_s < \bar{\alpha_s})\ \ \&\ \ (\alpha_o < \bar{\alpha_o})$ (Both are faster)

$(\alpha_s < \bar{\alpha_s})\ \ \&\ \ (\alpha_o > \bar{\alpha_o})$ (Self is fast, other is slow)

$(\alpha_s > \bar{\alpha_s})\ \ \&\ \ (\alpha_o < \bar{\alpha_o})$ (Self is slow, other is fast)

$(\alpha_s > \bar{\alpha_s})\ \ \&\ \ (\alpha_o > \bar{\alpha_o})$ (Both are slower)

In [7]:
df['self'].sum()

np.int64(11290)

In [8]:
dfo = pd.merge(
    left=df.loc[df['self']], left_on=['corpus', 'cid', 'speaker'],
    right=df[['corpus', 'cid', 'speaker', 'faster']].loc[~df['self']], right_on=['corpus', 'cid', 'speaker'],
    how='left'
)
print(dfo.shape)
dfo.isna().sum()

(11290, 10)


corpus       0
cid          0
speaker      0
self         0
a            0
b            0
k            0
b_           0
faster_x     0
faster_y    49
dtype: int64

In [9]:
dfo = dfo.loc[~dfo['faster_y'].isna()]
dfo.head()

,corpus,cid,speaker,self,a,b,k,b_,faster_x,faster_y
0,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-KB0PSUN,True,1.439414e-08,0.363082,136,-0.486513,False,True
1,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS002,True,1.493614e-03,-0.797152,2556,-0.486513,True,True
2,CABNC,CABNC-KB0RE000-KB0,CABNC-CABNC-KB0RE000-KB0-PS006,True,3.587024e-04,-0.430773,1953,-0.486513,False,True
3,CABNC,CABNC-KB0RE001-KB0,CABNC-CABNC-KB0RE001-KB0-KB0PSUN,True,5.573154e-06,-0.930984,105,-0.486513,True,True
4,CABNC,CABNC-KB0RE001-KB0,CABNC-CABNC-KB0RE001-KB0-PS005,True,5.494917e-04,0.090868,435,-0.486513,False,False


In [10]:
obs = np.array([
    ((dfo['faster_x']==True) & (dfo['faster_y']==True)).sum(),
    ((dfo['faster_x']==True) & (dfo['faster_y']==False)).sum(),
    ((dfo['faster_x']==False) & (dfo['faster_y']==True)).sum(),
    ((dfo['faster_x']==False) & (dfo['faster_y']==False)).sum(),
])
obs

array([3483, 1434, 2385, 3939])

In [11]:
obs/obs.sum()

array([0.30984788, 0.12756872, 0.21216974, 0.35041366])

In [12]:
exp = np.array([obs.mean()]*len(obs))
exp, exp.sum()

(array([2810.25, 2810.25, 2810.25, 2810.25]), np.float64(11241.0))

In [13]:
chisquare(f_obs=obs, f_exp=exp)

Power_divergenceResult(statistic=np.float64(1352.751801441153), pvalue=np.float64(5.266819521910178e-293))

## Hypothesis test: CANDOR is equivalent to the rest of the study

In [18]:
candor_obs = np.array([
    ((dfo['faster_x'].loc[dfo['corpus'].isin(['CANDOR'])]==True) & (dfo['faster_y'].loc[dfo['corpus'].isin(['CANDOR'])]==True)).sum(),
    ((dfo['faster_x'].loc[dfo['corpus'].isin(['CANDOR'])]==True) & (dfo['faster_y'].loc[dfo['corpus'].isin(['CANDOR'])]==False)).sum(),
    ((dfo['faster_x'].loc[dfo['corpus'].isin(['CANDOR'])]==False) & (dfo['faster_y'].loc[dfo['corpus'].isin(['CANDOR'])]==True)).sum(),
    ((dfo['faster_x'].loc[dfo['corpus'].isin(['CANDOR'])]==False) & (dfo['faster_y'].loc[dfo['corpus'].isin(['CANDOR'])]==False)).sum(),
])
candor_obs

array([1651,   77,  995,  589])

In [19]:
exp = (obs/obs.sum()) * candor_obs.sum()
exp

array([1026.21617294,  422.50760608,  702.70616493, 1160.57005604])

In [20]:
chisquare(f_obs=candor_obs,f_exp=exp)

Power_divergenceResult(statistic=np.float64(1065.9970814871297), pvalue=np.float64(8.667618682365903e-231))

## CANDOR conversation types yield different survey outcomes

In [14]:
df = dfo.loc[dfo['corpus'].isin(['CANDOR'])]

In [15]:
df.shape

(3312, 10)

In [16]:
df['cid'] = ['-'.join(it.split('-')[2:]) for it in df['cid']]

In [17]:
df.head()

,corpus,cid,speaker,self,a,b,k,b_,faster_x,faster_y
5499,CANDOR,142c436c-0d8f-49ee-9725-291486519f67,5eef7517c8692d20c3279493,True,0.001407,-0.457402,4656,-0.486513,False,True
5500,CANDOR,142c436c-0d8f-49ee-9725-291486519f67,5ef6bb87b83f3b000a3f04ae,True,0.003729,-0.690014,4753,-0.486513,True,False
5501,CANDOR,8bf5b973-cb87-4d3b-859b-8e116476295f,5f23ff398626c51fdd6c3fb5,True,0.000994,-0.557747,16110,-0.486513,True,True
5502,CANDOR,8bf5b973-cb87-4d3b-859b-8e116476295f,5f5eac716f1f7176f11d2bf0,True,0.000069,-0.257256,15931,-0.486513,False,False
5503,CANDOR,ea342a4f-891d-4717-8fb3-30bd1dadeca0,5b11682bd3ef5a000107dc58,True,0.000142,-0.323722,48609,-0.486513,False,False


### Merge with CANDOR survey data

In [18]:
vars = [
    ### Affective motive
    'best_affect', 'conv_leader', 'conversationalist', 'how_enjoyable', 'in_common', 'i_like_you', 'i_felt_close_to_my_partner', 'i_think_your_status', 'i_would_like_to_become_friends', 'overall_affect','responsive_1','responsive_2','responsive_3', 'you_are_friendly','you_are_good_listener', 'you_are_intelligent','you_are_kind', 'you_are_quickwitted','you_are_warm','you_like_me',

    ### Cognitive motives
    'anticipated_each_other_sr6', 'developed_joint_perspective_sr2', 'our_thoughts_synced_up_sr1', 'saw_world_in_same_way_sr8', 'shared_reality', 'thoughts_became_more_alike_sr5'
]

In [19]:
dfm = pd.read_csv(candor_meta_data_path)[['user_id', 'convo_id']+vars]
print(dfm.shape)
dfm.head()

(3312, 28)


,user_id,convo_id,best_affect,conv_leader,conversationalist,how_enjoyable,in_common,i_like_you,i_felt_close_to_my_partner,i_think_your_status,...,you_are_kind,you_are_quickwitted,you_are_warm,you_like_me,anticipated_each_other_sr6,developed_joint_perspective_sr2,our_thoughts_synced_up_sr1,saw_world_in_same_way_sr8,shared_reality,thoughts_became_more_alike_sr5
0,5efbe4f0c9f37c1a3b70afaf,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,7.0,6.0,76.0,7.0,7.0,6.0,NaN,8.0,...,9.0,6.0,9.0,6.0,5.0,7.0,5.0,4.0,5.375,6.0
1,5f26378609aca228dcfd19fa,56f21a75-4e86-4dee-9a15-a4f68e86e1a3,7.0,7.0,73.0,7.0,8.0,5.0,NaN,6.0,...,9.0,8.0,9.0,6.0,6.0,5.0,6.0,7.0,5.875,4.0
2,5d8e361e6e920e0014b459d9,94a1abaa-d89e-4d0d-82ad-1c8de8155a2f,7.0,3.0,33.0,5.0,1.0,4.0,NaN,4.0,...,2.0,2.0,2.0,4.0,1.0,1.0,1.0,2.0,2.375,2.0
3,5f4d435c030fef9d0136b496,94a1abaa-d89e-4d0d-82ad-1c8de8155a2f,5.0,8.0,65.0,4.0,3.0,5.0,NaN,6.0,...,9.0,9.0,9.0,5.0,1.0,2.0,2.0,1.0,2.000,1.0
4,5ca6bbf13b5fcf00100996e9,4e3233b7-5d89-47e9-a795-25452f99a1c9,9.0,4.0,96.0,9.0,6.0,7.0,6.0,6.0,...,9.0,8.0,9.0,7.0,5.0,3.0,4.0,4.0,4.500,5.0


#### Testing for normalcy

In [20]:
normalcy_and_other_stats = []

In [21]:
for var in vars:
    sel = dfm[var].isna()
    maxima = dfm[var].loc[~sel].max()
    minima = dfm[var].loc[~sel].min()
    medi = dfm[var].loc[~sel].median()
    mu = dfm[var].loc[~sel].mean()
    mode = dfm[var].loc[~sel].mode()
    skew_stat, p_skew = skewtest(dfm[var].loc[~sel].values)

    dv = dfm[var].loc[~sel].value_counts()
    chi_stat, p_chi = chisquare(f_obs=dv.values, f_exp=np.array([1/len(dv)]*len(dv))*dv.sum())

    normalcy_and_other_stats += [
        {
            'variable': var,
            'range': '[{}, {}]'.format(minima, maxima),
            'median': medi.item(),
            'mean': mu.item(),
            'mode': mode.values,
            'skew_stat': skew_stat, 'skew_p': p_skew,
            'chi_stat': chi_stat, 'chi_p': p_chi
        }
    ]

normalcy_and_other_stats = pd.DataFrame(normalcy_and_other_stats)

In [22]:
normalcy_and_other_stats.head(n=40)

,variable,range,median,mean,mode,skew_stat,skew_p,chi_stat,chi_p
0,best_affect,"[1.0, 9.0]",8.00,7.830826,[9.0],-20.435809,8.035056e-93,4714.587657,0.000000e+00
1,conv_leader,"[1.0, 9.0]",5.00,4.912803,[5.0],-0.177222,8.593339e-01,1118.964691,3.072156e-236
2,conversationalist,"[0.0, 100.0]",79.00,73.885477,[90.0],-22.179386,5.431503e-109,6551.635554,0.000000e+00
3,how_enjoyable,"[1.0, 9.0]",8.00,7.411115,[9.0],-22.780117,7.219819e-115,3218.084126,0.000000e+00
4,in_common,"[1.0, 9.0]",6.00,5.891004,[7.0],-10.400025,2.478679e-25,1141.369358,4.447494e-241
5,i_like_you,"[1.0, 7.0]",6.00,5.995088,[7.0],-23.672436,6.934913e-124,3718.741787,0.000000e+00
6,i_felt_close_to_my_partner,"[1.0, 7.0]",5.00,4.909976,[6.0],-8.470986,2.433243e-17,341.532847,1.013785e-70
7,i_think_your_status,"[1.0, 10.0]",6.00,5.911548,[5.0],-1.936587,5.279590e-02,2895.382064,0.000000e+00
8,i_would_like_to_become_friends,"[1.0, 7.0]",5.00,4.970803,[6.0],-7.628042,2.383459e-14,254.569343,4.328281e-52
9,overall_affect,"[1.0, 9.0]",7.00,7.323611,[7.0],-13.989633,1.803392e-44,3368.975745,0.000000e+00


In [23]:
normalcy_and_other_stats.to_csv('candor-normalcy-stats.csv', index=False, encoding='utf-8')

#### Merging docs

In [20]:
df = pd.merge(
    left=df, left_on=['speaker', 'cid'],
    right=dfm, right_on=['user_id', 'convo_id'],
    # left=results, left_on=['speaker'],
    # right=dfm, right_on=['user_id'],
    how='left'
)

In [21]:
df = df.drop(columns=['user_id', 'convo_id'])
# results = results.drop(columns=['user_id'])
# results.columns
df.isna().sum()

corpus                                0
cid                                   0
speaker                               0
self                                  0
a                                     0
b                                     0
k                                     0
b_                                    0
faster_x                              0
faster_y                              0
best_affect                          55
conv_leader                          55
conversationalist                    55
how_enjoyable                        55
in_common                            55
i_like_you                           55
i_felt_close_to_my_partner         2490
i_think_your_status                  56
i_would_like_to_become_friends     2490
overall_affect                       55
responsive_1                         56
responsive_2                         56
responsive_3                         56
you_are_friendly                     55
you_are_good_listener                55


## Tests

##### Convert values to ints for intercepts

In [22]:
## convert speaker to integers
col = 'speaker'
vs = df[col].unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
df[col+'_'] = [sid[sp] for sp in tqdm(df[col])]

  0%|          | 0/3312 [00:00<?, ?it/s]

In [23]:
## convert convo_id col to integers
col = 'cid'
vs = df[col].unique()
# sid = {sp:i+1 for i,sp in enumerate(np.random.choice(vs,size=(len(vs),),replace=False))}
sid = {sp:i+1 for i,sp in enumerate(vs)}
df[col+'_'] = [sid[sp] for sp in tqdm(df[col])]

  0%|          | 0/3312 [00:00<?, ?it/s]

### Model in R

In [24]:
def random_effects_df(x):
    output = np.array(str(x).split()[:-2]).reshape(-1,3)
    odf = pd.DataFrame()
    odf['se'] = output[1:,-1].astype(float)
    odf.index = output[1:,0]
    return odf

In [25]:
##########################################
## A set of resids to show ∇H / t
##########################################
# formula = "b ~ v_s + v_o + (1|self) + (1|speaker_)"
# formula = "b ~ v_s + self + v_o + other + (1|speaker_) + (1|cid_) -1"
formula = "{} ~ faster_x + faster_y"
# model = "Hxy ~ nx + ny"
# model = "Hxy ~ nx + ny + null"
##########################################

In [26]:
df_all = []

In [27]:
for var in vars:
    sel_2 = ~df[var].isna()

    FO = Formula(formula.format(var))
    # model_ = str(model_script)

    with localconverter(default_converter + pandas2ri.converter):
        df_ = conversion.py2rpy(df[[var, 'faster_x', 'faster_y', 'speaker_', 'cid_']].loc[sel_2])

    ro.globalenv['df'] = df_
    ro.globalenv['formula'] = FO

    ro.r("""
    library(lme4)
    print(colnames(df))

    # model <- lmer(formula, data=df)
    model <- glm(formula, data=df)
    summarization <- summary(model)
    """)

    summary = ro.r('summarization')

    # ro.r('cov_vals <- vcov(model)')
    # covar = ro.r('cov_vals')
    # covar_df = pd.DataFrame(pandas2ri.rpy2py(covar))
    # covar_df.index = list(covar.names[0])
    # covar_df.columns = list(covar.names[1])
    # covar_df.head(n=10)
    #
    # re_effects_df = random_effects_df(summary.rx2('varcor'))
    # re_effects_df.index = ['(1|{})'.format(idx) for idx in re_effects_df.index]

    fixed_effects = summary.rx2('coefficients')
    fixed_effects_df = pd.DataFrame(pandas2ri.rpy2py(fixed_effects))
    fixed_effects_df.index = list(fixed_effects.names[0])
    fixed_effects_df.columns = list(fixed_effects.names[1])
    fixed_effects_df = fixed_effects_df.rename(columns={'Std. Error': 'se'})



    df_params = pd.concat([
        fixed_effects_df,
        # re_effects_df,
        # covar_df
    ],axis=1)
    df_params['k'] = sel_2.sum()
    # df_params['p'] = [t.sf(tstat.__abs__(), df=(k-1)) for tstat, k in df_params[['t value', 'k']].values]
    df_params['var'] = df_params.index.values
    df_params['survey_var'] = var

    df_all += [df_params]
df_all = pd.concat(df_all,ignore_index=True)

# sort and preamble
preambulatory = ['survey_var', 'var']
df_all = df_all[preambulatory+[col for col in list(df_all) if col not in preambulatory]]

R callback write-console: Loading required package: Matrix
  


[1] "best_affect" "faster_x"    "faster_y"    "speaker_"    "cid_"       
[1] "conv_leader" "faster_x"    "faster_y"    "speaker_"    "cid_"       
[1] "conversationalist" "faster_x"          "faster_y"         
[4] "speaker_"          "cid_"             
[1] "how_enjoyable" "faster_x"      "faster_y"      "speaker_"     
[5] "cid_"         
[1] "in_common" "faster_x"  "faster_y"  "speaker_"  "cid_"     
[1] "i_like_you" "faster_x"   "faster_y"   "speaker_"   "cid_"      
[1] "i_felt_close_to_my_partner" "faster_x"                  
[3] "faster_y"                   "speaker_"                  
[5] "cid_"                      
[1] "i_think_your_status" "faster_x"            "faster_y"           
[4] "speaker_"            "cid_"               
[1] "i_would_like_to_become_friends" "faster_x"                      
[3] "faster_y"                       "speaker_"                      
[5] "cid_"                          
[1] "overall_affect" "faster_x"       "faster_y"       "speaker_"      

In [28]:
df_all.head(n=50)

,survey_var,var,Estimate,se,t value,Pr(>|t|),k
0,best_affect,(Intercept),7.961831,0.046454,171.390426,0.000000e+00,3257
1,best_affect,faster_xTRUE,-0.103887,0.045363,-2.290149,2.207640e-02,3257
2,best_affect,faster_yTRUE,-0.095989,0.056527,-1.698108,8.958306e-02,3257
3,conv_leader,(Intercept),5.060153,0.078449,64.502529,0.000000e+00,3257
4,conv_leader,faster_xTRUE,-0.277929,0.076605,-3.628058,2.899482e-04,3257
5,conv_leader,faster_yTRUE,-0.002539,0.095459,-0.026602,9.787791e-01,3257
6,conversationalist,(Intercept),76.315014,0.788217,96.819860,0.000000e+00,3257
7,conversationalist,faster_xTRUE,-2.147195,0.769693,-2.789678,5.306764e-03,3257
8,conversationalist,faster_yTRUE,-1.635790,0.959127,-1.705500,8.819670e-02,3257
9,how_enjoyable,(Intercept),7.580499,0.060148,126.031149,0.000000e+00,3257


In [29]:
df_all.to_csv('regression_estimates.csv', index=False, encoding='utf-8')